In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path
import pandas as pd
from form990_parser import (
    parse_header,
    parse_return,
    get_org_type
)

In [2]:
%load_ext autoreload
%autoreload 2

In [ ]:
xml_data_directory = Path(r'xml')
year_dir = xml_data_directory / '2024'
period_dir = year_dir / '2024_01A'

In [10]:
%%time
counter = 0
for xml_file in period_dir.glob('*.xml'):
    # Skip certain orgs and form types
    root = ET.parse(xml_file).getroot()
    org_type = get_org_type(root)
    if org_type not in ['501c3', '501c4', '527']:
        continue 

    if counter >= 50:
        break
    counter += 1

    # * Header
    header_content = parse_header(root, xml_file, org_type)

    # * Return
    if parse_return(root, 'tbd') is None:
        continue # Some files pertain to other types of returns (e.g., 990-T)
    return_content, employee_content, contractor_content, expense_content, balance_sheet_content = parse_return(root, 'tbd')

CPU times: total: 359 ms
Wall time: 1.7 s


In [23]:
%%timeit
root = ET.parse(xml_file).getroot() # 712e-6

742 μs ± 27.5 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [ ]:
%%timeit
for elem in root.iter(): 
    if elem.tag 